# Baseline Retrieval vs. Cross-Encoder Re-Ranking

Two retrieval pipelines over **one** chunked corpus — the *Twilight Imperium 4th Edition
Living Rules Reference* (v1.3, official FFG PDF).

* **Pipeline 1 (baseline)** — bi-encoder embeddings in a FAISS index → **Top 10** by cosine
  similarity → **Top 3** straight into the prompt → **Answer 1**.
* **Pipeline 2 (re-ranking)** — the **same Top 10** → all 10 `(query, chunk)` pairs scored by
  a cross-encoder → re-sorted → **Top 3** → the **identical prompt template** → **Answer 2**.

Everything downstream (rank shifts, answer comparison, latency, scaling) is written to
`results/` as CSV so the report can cite a file rather than a screenshot.
Design decisions and their justifications live in `NOTES.md`.

In [1]:
# =============================================================================================
# Configuration. Every knob in the notebook lives here.
# =============================================================================================
from pathlib import Path
import hashlib, json, os, platform, re, statistics, time

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, RESULTS = ROOT / "data", ROOT / "results"
DATA.mkdir(exist_ok=True); RESULTS.mkdir(exist_ok=True)

# Corpus: the official Fantasy Flight Games PDF. Gitignored, so the notebook fetches it.
PDF_URL  = ("https://images-cdn.fantasyflightgames.com/filer_public/da/df/"
            "dadf9f07-78f3-43ac-9cce-dd6b55b24ec2/ti4_living_rules_reference_v1_3_web.pdf")
PDF_PATH = DATA / "ti4_living_rules_reference_v1_3_web.pdf"

# Models. Bi-encoder and cross-encoder are both 6-layer MiniLM, i.e. comparable size — but NOT
# a controlled architecture-only comparison, since they differ in training data and objective
# as well (see NOTES.md 1.7).
BI_ENCODER_NAME    = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Generation model. ONE config variable: model ids expire (gemini-2.5-flash passed its
# 2026-10-16 shutdown), so the smoke test below resolves the first candidate that responds.
GEN_MODEL_CANDIDATES = ["gemini-3.5-flash", "gemini-3.1-flash-lite"]
GEN_MODEL = None                      # filled in by the smoke test

# Retrieval sizes are fixed by the assignment.
TOP_K_RETRIEVE = 10
TOP_K_CONTEXT  = 3

# Chunking. 256 word-pieces is a hard truncation limit for all-MiniLM-L6-v2; staying under it is
# asserted, not assumed, because silent truncation would look exactly like a retrieval bug.
MAX_CHUNK_TOKENS   = 230
TOKENIZER_HARD_CAP = 256
OVERLAP_UNITS      = 1                # one trailing rule repeated into the next chunk

# LLM budget. Every answer is cached on disk and never regenerated, so re-running this notebook
# costs zero API calls. Raised from 18 to 22 after the first run: eight hand-written queries
# yielded only ONE clean promotion (five had the gold chunk at rank 1-2 already, so re-ranking
# had nothing to repair), and the assignment requires at least two. Q9 and Q10 were found by the
# retrieval-only hunt in §4b, which costs no quota, and adding their two answers each is what
# took the ceiling to 22. The observed limit is per-minute, not per-day — see NOTES.md 3.6.
LLM_CALL_BUDGET = 22
CACHE_PATH      = RESULTS / "answers_cache.json"

NEAR_DUP_COSINE = 0.90                # threshold for logging overlap-induced near-duplicates
LATENCY_REPEATS = 10                  # timed repeats after a discarded warm-up pass

print(f"root={ROOT}")
print(f"python={platform.python_version()}  machine={platform.machine()}  "
      f"system={platform.system()} {platform.release()}")

root=/Users/d400l/Desktop/LMA_RAG/LMA_RAG
python=3.12.12  machine=arm64  system=Darwin 25.4.0


In [2]:
# Fetch the corpus if it is not already present (data/*.pdf is gitignored).
import urllib.request

if not PDF_PATH.exists():
    print("downloading corpus ...")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
print(f"{PDF_PATH.name}: {PDF_PATH.stat().st_size:,} bytes")

ti4_living_rules_reference_v1_3_web.pdf: 3,313,611 bytes


In [3]:
# Load both local models and report the device actually in use (needed for the latency table).
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
bi_encoder    = SentenceTransformer(BI_ENCODER_NAME, device=DEVICE)
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device=DEVICE)

HARDWARE = (f"{platform.machine()} / {platform.system()} {platform.release()} / "
            f"torch {torch.__version__} / device={DEVICE}")
print(HARDWARE)
print("bi-encoder max_seq_length:", bi_encoder.max_seq_length,
      "| embedding dim:", bi_encoder.get_sentence_embedding_dimension())

def count_tokens(text: str) -> int:
    """Word-piece length according to the bi-encoder's OWN tokenizer, special tokens included.
    Character or word heuristics are not good enough here: the model truncates at 256
    word-pieces, and anything past that is invisible to retrieval while still visible to the
    LLM in the prompt — a bug that would present as a retrieval failure."""
    return len(bi_encoder.tokenizer(text)["input_ids"])

/Users/d400l/Desktop/LMA_RAG/LMA_RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


arm64 / Darwin 25.4.0 / torch 2.5.1 / device=mps
bi-encoder max_seq_length: 256 | embedding dim: 384


In [4]:
# =============================================================================================
# Generation client.
#
# Temperature is deliberately LEFT UNSET. Google's Gemini 3 guidance strongly recommends not
# overriding temperature/top_p/top_k because the models are tuned around their defaults, and
# google-genai sends no temperature at all while the field is None. The "only the retrieved
# context differs" claim is therefore carried by (a) a byte-identical prompt template, asserted
# in §5, and (b) a persistent answer cache that freezes each generation on disk.
#
# Thinking is set to MINIMAL: §9 asks for latency numbers, and a variable amount of hidden
# reasoning would add variance to the one quantity being tabulated.
# =============================================================================================
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv(ROOT / ".env")
assert os.getenv("GEMINI_API_KEY"), "GEMINI_API_KEY missing — copy .env.example to .env"
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])      # the key is never printed

# thinking_level="MINIMAL" needs a recent SDK and model; the call below falls back to an empty
# config if either rejects it, and remembers which variant worked.
THINKING_CFG = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(thinking_level="MINIMAL"))
PLAIN_CFG    = types.GenerateContentConfig()

LLM_CALLS  = 0            # counts SUCCESSFUL generations only; capped by LLM_CALL_BUDGET
_exhausted = set()        # models whose quota is spent; never probed again this run
GEN_CONFIG = None         # frozen after the first success, so probing costs at most one request
RPM_PAUSE_S     = 4.0     # minimum spacing between requests
MAX_RETRY       = 3       # short: a genuinely exhausted model should fail over, not be waited on
MAX_WAIT_S      = 30.0    # cap one backoff, so an exhausted model cannot stall the run
_last_call_ts   = 0.0
# Transient conditions worth waiting out rather than failing the run. 429 is the per-minute
# rate limit; 5xx/UNAVAILABLE is the shared-capacity "high demand" response, which the first
# ten-query run hit on a 503 and which has nothing to do with quota.
TRANSIENT = ("429", "RESOURCE_EXHAUSTED", "500", "502", "503", "504",
             "UNAVAILABLE", "overloaded", "high demand", "INTERNAL")

def _retry_delay(msg: str, attempt: int) -> float:
    """Honour the server's retryDelay when it sends one, otherwise back off linearly. Capped:
    waiting ten minutes on a model whose daily quota is gone is not patience, it is a hang."""
    m = re.search(r'retryDelay[\"\':\s]+(\d+(?:\.\d+)?)s', msg)
    return min(float(m.group(1)) + 1 if m else 15.0 * (attempt + 1), MAX_WAIT_S)

def _is_quota_exhaustion(msg: str) -> bool:
    """A 429 means two different things. "exceeded your current quota" is a spent allowance that
    no amount of waiting fixes within this run, so the right response is to fail over to the
    next candidate model. A plain per-minute rate limit is worth retrying."""
    return "exceeded your current quota" in msg or "quota_metric" in msg

def _raw_call(model: str, prompt: str):
    """One generation request -> (text, request_seconds).

    The free tier here limits requests PER MINUTE, not per day: the first run tripped a 429
    after six requests in quick succession and recovered within minutes. Transient failures
    (429 rate limits and 5xx "high demand") are retried with backoff against the SAME model —
    switching models mid-run would make Answer 1 and Answer 2 incomparable — and requests are
    spaced at least RPM_PAUSE_S apart.

    Two details that keep the measurements honest: only successful generations count against
    LLM_CALL_BUDGET, and the returned duration covers the successful request alone, so waiting
    out a rate limit can never leak into the latency table.
    """
    global LLM_CALLS, GEN_CONFIG, _last_call_ts
    if model in _exhausted:
        raise RuntimeError(f"{model}: quota already known exhausted this run")
    variants = [GEN_CONFIG] if GEN_CONFIG else [("minimal-thinking", THINKING_CFG),
                                                ("default-thinking", PLAIN_CFG)]
    errors = []
    for name, cfg in variants:
        for attempt in range(MAX_RETRY + 1):
            if LLM_CALLS >= LLM_CALL_BUDGET:
                raise RuntimeError(f"LLM call budget of {LLM_CALL_BUDGET} exhausted")
            gap = RPM_PAUSE_S - (time.time() - _last_call_ts)
            if gap > 0:
                time.sleep(gap)
            t0 = time.perf_counter()
            try:
                r = client.models.generate_content(model=model, contents=prompt, config=cfg)
                dt = time.perf_counter() - t0
                _last_call_ts = time.time()
                LLM_CALLS += 1
                GEN_CONFIG = (name, cfg)
                return (r.text or "").strip(), dt
            except Exception as e:
                _last_call_ts = time.time()
                msg = str(e)
                if _is_quota_exhaustion(msg):
                    _exhausted.add(model)          # no point confirming it a second time
                    raise RuntimeError(f"quota exhausted for {model}") from None
                if any(t in msg for t in TRANSIENT):
                    wait = _retry_delay(msg, attempt)
                    print(f"    transient failure on {model} ({type(e).__name__}); waiting "
                          f"{wait:.0f}s (attempt {attempt + 1}/{MAX_RETRY + 1})")
                    time.sleep(wait)
                    continue
                errors.append(f"{name}: {type(e).__name__}: {msg[:160]}")
                break                     # a real error, not transient: try the next variant
        else:
            errors.append(f"{name}: still failing after {MAX_RETRY + 1} attempts")
    raise RuntimeError("generation failed:\n  " + "\n  ".join(errors))

# Smoke test: resolve GEN_MODEL to the first candidate that actually answers.
for candidate in GEN_MODEL_CANDIDATES:
    try:
        reply, _ = _raw_call(candidate, "Reply with the single word: ready")
        GEN_MODEL = candidate
        print(f"GEN_MODEL={GEN_MODEL!r} [{GEN_CONFIG[0]}] replied {reply[:40]!r}")
        break
    except Exception as e:
        print(f"{candidate!r} unavailable: {str(e)[:220]}")
assert GEN_MODEL, "no candidate generation model responded"
print(f"temperature sent: {THINKING_CFG.temperature}  (None = provider default)")
print(f"LLM calls used by setup: {LLM_CALLS}/{LLM_CALL_BUDGET}")


def advance_model() -> bool:
    """Move GEN_MODEL to the next candidate that still answers. Used when a model's quota runs
    out mid-run: the assignment says to use whichever candidate responds, and mid-run
    exhaustion is exactly that situation arriving late."""
    global GEN_MODEL
    rest = GEN_MODEL_CANDIDATES[GEN_MODEL_CANDIDATES.index(GEN_MODEL) + 1:]
    for cand in rest:
        try:
            _raw_call(cand, "Reply with the single word: ready")
            print(f"  failed over: GEN_MODEL {GEN_MODEL!r} -> {cand!r}")
            GEN_MODEL = cand
            return True
        except Exception as e:
            print(f"  candidate {cand!r} also unavailable: {str(e)[:120]}")
    return False

'gemini-3.5-flash' unavailable: quota exhausted for gemini-3.5-flash


GEN_MODEL='gemini-3.1-flash-lite' [minimal-thinking] replied 'ready'
temperature sent: None  (None = provider default)
LLM calls used by setup: 1/22


## 1. Corpus and chunking

The Living Rules Reference is a two-column, alphabetised rules *reference*: almost every
statement carries a rule number (`58.6`, `2.4`), which gives a semantically meaningful split
point that no generic character-window splitter would find. Chunks therefore follow rule
boundaries, and each one records its rule numbers and pages so every retrieved chunk is
citable in the report as "Rule 58.6, p.21".

Three preprocessing decisions, all visible in the code below:

1. **Pages 6–31 only** — the numbered rules. The table of contents, the setup walkthrough and
   the back-of-book index are excluded: they are navigation, not rule content, and the index
   in particular is a list of bare keywords plus rule numbers that would inject lexically
   matching but semantically empty chunks, manufacturing "keyword traps" that are artifacts
   of preprocessing rather than properties of the document.
2. **Geometric block filter** — PyMuPDF returns card illustrations and duplicated summary
   diagrams interleaved with body prose, and returns the right-hand column *before* the left
   on some pages. Blocks are kept by column margin/width and re-sorted column-major.
3. **`RELATED TOPICS:` lines are dropped** — they are cross-reference keyword lists. Keeping
   them would act as an artificial lexical magnet and distort the very retrieval comparison
   this notebook is measuring.

In [5]:
import fitz   # PyMuPDF

RULES_PAGES  = range(5, 31)                        # 0-indexed; printed pages 6-31
BODY_MARGINS = (36.0, 54.0, 315.0, 333.0)          # left body, left bullet, right body, right bullet
COL_SPLIT    = 306.0

_LEAD = re.compile(r'^(\d{1,2}(?:\.\d{1,2}[a-z]?)?)\t\s*(.*)$', re.S)

def _norm(t: str) -> str:
    """Normalise one PDF block. Tabs are protected because the rule number is separated from
    its text by a tab, and collapsing it would destroy the only structural signal available."""
    t = t.replace(' ', ' ').replace('\xa0', ' ').replace(' ', ' ')
    t = t.replace('\t', '\x01')
    t = re.sub(r'\s*\n\s*', ' ', t)                # line breaks are layout, not meaning
    t = re.sub(r'[ ]+', ' ', t).strip()
    return t.replace('\x01', '\t')

def body_blocks(page):
    """Body prose only, in true reading order (left column top-to-bottom, then right)."""
    out = []
    for x0, y0, x1, y1, txt, *_ in page.get_text("blocks"):
        if y0 > 740 or y1 < 45 or not txt.strip():        # footer / header
            continue
        in_col      = (30 <= x0 <= 300) or (310 <= x0 <= 580)
        near_margin = any(abs(x0 - m) <= 4 for m in BODY_MARGINS)
        if in_col and (near_margin or (x1 - x0) >= 120):  # narrow stray blocks are card art
            out.append((0 if x0 < COL_SPLIT else 1, y0, _norm(txt)))
    out.sort(key=lambda r: (r[0], r[1]))
    return [t for _, _, t in out]

def parse_units(doc):
    """Parse the rules into units: one numbered rule plus its bullets and wrapped lines.
    A section's unnumbered preamble becomes `<section>.0`, matching the document's own index
    convention (the back index refers to section preambles as e.g. "agenda phase, 8.0")."""
    units, stats = [], {"related_topics_dropped": 0, "orphan_blocks": 0}
    sec = title = cur = None

    def flush():
        nonlocal cur
        if cur and cur["text"].strip():
            units.append(cur)
        cur = None

    for pno in RULES_PAGES:
        for txt in body_blocks(doc[pno]):
            if txt.startswith("RELATED TOPICS"):
                stats["related_topics_dropped"] += 1
                continue
            m = _LEAD.match(txt)
            if m and "." not in m.group(1):                      # "9  ANOMALIES"
                flush()
                sec, title = m.group(1), (m.group(2).strip() or f"SECTION {m.group(1)}")
            elif m:                                              # "9.1  An anomaly is ..."
                flush()
                cur = {"sec": sec, "title": title, "rule_no": m.group(1),
                       "text": m.group(2).replace("\t", " ").strip(), "page": pno + 1}
            elif cur is not None:                                # bullet or continuation
                cur["text"] += " " + txt.replace("\t", " ")
            elif sec is not None:                                # section preamble
                cur = {"sec": sec, "title": title, "rule_no": f"{sec}.0",
                       "text": txt.replace("\t", " "), "page": pno + 1}
            else:
                stats["orphan_blocks"] += 1                      # front matter before Rule 1
    flush()
    return units, stats

doc = fitz.open(PDF_PATH)
units, parse_stats = parse_units(doc)
print(f"pages={doc.page_count}  rule units={len(units)}  "
      f"sections={len({(u['sec'], u['title']) for u in units})}  {parse_stats}")

pages=38  rule units=532  sections=89  {'related_topics_dropped': 89, 'orphan_blocks': 2}


In [6]:
def _rule_key(r):
    a, _, b = r.partition(".")
    return (int(a), int(re.sub(r'\D', '', b) or 0), b)

def build_chunks(units, max_tokens=MAX_CHUNK_TOKENS, overlap=OVERLAP_UNITS):
    """Pack consecutive rule units of the same section into chunks under the token cap.

    Two details that matter for the experiment:
    * Each chunk is prefixed with its section title ("ABILITIES (Rule 1)"). Sub-rules are often
      one short clause whose subject lives in the heading, so without the prefix the embedding
      of "9.1 An anomaly is identified by a red border" carries no topical anchor at all.
    * `overlap=1` repeats the last rule of a chunk as the first rule of the next, so a rule and
      its immediate context stay together across a boundary. The cost is near-duplicate
      candidates, which §4 logs explicitly rather than hiding."""
    by_sec = {}
    for u in units:
        by_sec.setdefault((u["sec"], u["title"]), []).append(u)

    def render(sec, title, group):
        body = "\n".join(f'{u["rule_no"]} {u["text"]}' for u in group)
        return f"{title} (Rule {sec})\n{body}"

    def split_oversize(sec, title, u):
        """One rule longer than the cap is divided at sentence boundaries, keeping its rule
        number so gold-chunk matching still works."""
        if count_tokens(render(sec, title, [u])) <= max_tokens:
            return [u]
        parts, buf = [], ""
        for sent in re.split(r'(?<=[.!?]) +', u["text"]):
            trial = (buf + " " + sent).strip()
            if buf and count_tokens(render(sec, title, [dict(u, text=trial)])) > max_tokens:
                parts.append(dict(u, text=buf)); buf = sent
            else:
                buf = trial
        if buf:
            parts.append(dict(u, text=buf))
        return parts

    packed = []
    for (sec, title), us in by_sec.items():
        us = [p for u in us for p in split_oversize(sec, title, u)]
        i, group = 0, []
        while i < len(us):
            if group and count_tokens(render(sec, title, group + [us[i]])) > max_tokens:
                packed.append((sec, title, list(group)))
                tail = group[-overlap:] if overlap else []
                if tail and count_tokens(render(sec, title, tail + [us[i]])) > max_tokens:
                    tail = []                                    # overlap itself would overflow
                group = tail
                continue
            group = group + [us[i]]
            i += 1
        if group:
            packed.append((sec, title, list(group)))

    rows = []
    for sec, title, group in packed:
        text = render(sec, title, group)
        rows.append({"section_no": int(sec), "section_title": title,
                     "rule_from": group[0]["rule_no"], "rule_to": group[-1]["rule_no"],
                     "rule_nos": ";".join(u["rule_no"] for u in group),
                     "page_from": min(u["page"] for u in group),
                     "page_to": max(u["page"] for u in group),
                     "n_units": len(group), "n_tokens": count_tokens(text), "text": text})
    rows.sort(key=lambda r: (r["section_no"], _rule_key(r["rule_from"])))
    for cid, r in enumerate(rows):
        r["chunk_id"] = cid
    return rows

import pandas as pd

chunk_rows = build_chunks(units)
chunks = pd.DataFrame(chunk_rows)[["chunk_id", "section_no", "section_title", "rule_from",
                                   "rule_to", "rule_nos", "page_from", "page_to", "n_units",
                                   "n_tokens", "text"]]

# Hard assertions, not hopeful comments: nothing may reach the 256 word-piece truncation point,
# and chunk_id must coincide with the row index because every lookup below uses `.loc[chunk_id]`.
assert chunks["n_tokens"].max() < TOKENIZER_HARD_CAP, "a chunk would be silently truncated"
assert (chunks.index == chunks["chunk_id"]).all(), "chunk_id must equal the row index"
chunks.to_csv(RESULTS / "chunks.csv", index=False)

print(f"chunks={len(chunks)}  tokens: min={chunks.n_tokens.min()} "
      f"median={int(chunks.n_tokens.median())} max={chunks.n_tokens.max()} "
      f"(hard cap {TOKENIZER_HARD_CAP})")
print(chunks.head(3)[["chunk_id", "section_title", "rule_from", "rule_to", "page_from", "n_tokens"]]
      .to_string(index=False))
print("\nexample chunk:\n" + chunks.loc[0, "text"][:400])

Token indices sequence length is longer than the specified maximum sequence length for this model (267 > 256). Running this sequence through the model will result in indexing errors


chunks=172  tokens: min=20 median=179 max=229 (hard cap 256)
 chunk_id section_title rule_from rule_to  page_from  n_tokens
        0     ABILITIES       1.0     1.6          6       211
        1     ABILITIES       1.6    1.11          6       185
        2     ABILITIES      1.11    1.15          6       200

example chunk:
ABILITIES (Rule 1)
1.0 Cards and faction sheets each have abilities that players can resolve to trigger various game effects.
1.1 Each ability describes when and how a player can resolve it. •  If an ability with a specified duration is resolved, the effect of the ability remains through that duration, even if the component that caused the ability is removed.
1.2 If a card has multiple abilities, 


## 2. Pipeline 1 — baseline retrieval (bi-encoder)

Chunks are embedded independently, L2-normalised, and stored in a FAISS `IndexFlatIP`. Inner
product over L2-normalised vectors *is* cosine similarity, which satisfies the assignment's
"cosine similarity or inner product" either way.

`IndexFlatIP` is **exact** brute-force search, chosen over an approximate index (IVF/HNSW) on
purpose: under an ANN index a gold chunk missing from the Top 10 could be blamed on
approximation error instead of on bi-encoder semantics, which would make every failure-mode
claim below unfalsifiable.

In [7]:
import faiss, numpy as np

corpus_texts = chunks["text"].tolist()
corpus_emb = bi_encoder.encode(corpus_texts, batch_size=64, convert_to_numpy=True,
                               normalize_embeddings=True, show_progress_bar=False)
index = faiss.IndexFlatIP(corpus_emb.shape[1])
index.add(corpus_emb.astype(np.float32))
print(f"indexed {index.ntotal} chunks, dim={corpus_emb.shape[1]}, metric=inner product "
      f"on L2-normalised vectors (= cosine)")

def baseline_retrieve(query: str, k: int = TOP_K_RETRIEVE):
    """Pipeline 1 retrieval. Returns the ranked candidate list that BOTH pipelines consume."""
    q = bi_encoder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, ids = index.search(q.astype(np.float32), k)
    return [{"chunk_id": int(c), "bi_score": float(s), "bi_rank": r + 1}
            for r, (c, s) in enumerate(zip(ids[0], scores[0]))]

indexed 172 chunks, dim=384, metric=inner product on L2-normalised vectors (= cosine)


In [8]:
# ---------------------------------------------------------------------------------------------
# The prompt. ONE template, used by both pipelines — the only thing that changes between
# Answer 1 and Answer 2 is which three chunks get interpolated, and that is asserted later.
# "Answer only from the excerpts" matters: without it the model can paper over a missing chunk
# from prior knowledge of the game, and an omission caused by retrieval would never surface.
# ---------------------------------------------------------------------------------------------
PROMPT_TEMPLATE = """You are answering a rules question about the board game Twilight Imperium \
(4th Edition). Use ONLY the numbered rule excerpts below. Do not use outside knowledge. If the \
excerpts do not contain the answer, reply exactly: "The provided rules do not cover this."

RULE EXCERPTS
{context}

QUESTION: {question}

Answer in at most four sentences, and cite the rule numbers you relied on."""

def build_prompt(question: str, chunk_ids) -> str:
    ctx = "\n\n".join(f"[excerpt {i}]\n{corpus_texts[cid]}" for i, cid in enumerate(chunk_ids, 1))
    return PROMPT_TEMPLATE.format(context=ctx, question=question)

# Persistent answer cache keyed by (model, sha256(prompt)). An answer is generated once and
# then frozen on disk, so the committed analysis can never drift from the committed answers and
# re-running the notebook costs zero API quota.
_cache = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}

def _cache_key(model: str, prompt: str) -> str:
    return hashlib.sha256(f"{model}\x00{prompt}".encode()).hexdigest()

def _cache_lookup(prompt: str, strict: bool = False):
    """Look for this prompt under the current model first, then under the other candidates.

    This matters because a model can run out of quota mid-project: answers generated earlier on
    one model must stay usable instead of being silently regenerated on another, which would
    both burn quota and make Answer 1 and Answer 2 come from different models within one query.
    The entry records which model produced it, and the run loop asserts each query's pair agrees."""
    order = [GEN_MODEL] if strict else \
            [GEN_MODEL] + [m for m in GEN_MODEL_CANDIDATES if m != GEN_MODEL]
    for m in order:
        k = _cache_key(m, prompt)
        if k in _cache:
            return k, _cache[k]
    return None, None

def generate(question: str, chunk_ids, strict_model: bool = False):
    """-> (answer, prompt, cache_hit, seconds, model). Identical prompts collapse onto one cache
    entry, so when re-ranking leaves the Top 3 set unchanged, Answer 2 IS Answer 1, by
    construction."""
    prompt = build_prompt(question, chunk_ids)
    hit_key, hit = _cache_lookup(prompt, strict=strict_model)
    if hit:
        return hit["answer"], prompt, True, 0.0, hit["model"]
    key = _cache_key(GEN_MODEL, prompt)
    try:
        answer, dt = _raw_call(GEN_MODEL, prompt)   # dt excludes any waiting
    except RuntimeError:
        if not advance_model():                     # quota gone: use whichever model responds
            raise
        key = _cache_key(GEN_MODEL, prompt)
        answer, dt = _raw_call(GEN_MODEL, prompt)
    _cache[key] = {"model": GEN_MODEL, "cache_key": key, "answer": answer,
                   "n_context_chunks": len(chunk_ids), "seconds": round(dt, 3)}
    CACHE_PATH.write_text(json.dumps(_cache, indent=2))
    return answer, prompt, False, dt, GEN_MODEL

## 3. Pipeline 2 — cross-encoder re-ranking

The cross-encoder receives the **same 10 candidates** the baseline produced — it never
re-retrieves — scores every `(query, chunk)` pair with joint self-attention over both texts,
and the list is re-sorted by that score. Scores are raw relevance logits, comparable *within*
one query's candidate list (all that re-sorting needs) but not across queries, so no absolute
threshold is used anywhere.

In [9]:
def rerank(query: str, candidates):
    """Pipeline 2. Consumes the baseline candidate list and returns it re-sorted."""
    pairs = [(query, corpus_texts[c["chunk_id"]]) for c in candidates]
    ce_scores = cross_encoder.predict(pairs, show_progress_bar=False)
    out = [dict(c, ce_score=float(s)) for c, s in zip(candidates, ce_scores)]
    out.sort(key=lambda c: -c["ce_score"])
    for r, c in enumerate(out, 1):
        c["ce_rank"] = r
    return out

## 4. Query set

Eight questions, hand-written after reading the rulebook, each with its **gold rule numbers**
recorded before any retrieval was run. They are deliberately phrased the way a player would
ask — paraphrases and synonyms rather than the rulebook's own section vocabulary — because a
query that quotes the heading verbatim tests nothing.

The categories are the ones named in the assignment. Whether a category actually *produces* a
baseline failure in this document is an empirical result, reported as found: promotions,
no-ops and regressions all stay in the table.

In [10]:
QUERIES = [
    {"qid": "Q1", "category": "keyword_trap",
     "question": "How many cards is a player allowed to keep in their hand between rounds?",
     "gold_rules": ["2.4"],
     "why": "'cards' and 'hand' collide with promissory-note and objective-card passages; the "
            "hand limit itself lives in ACTION CARDS and never uses the word 'round'."},

    {"qid": "Q2", "category": "synonym_gap",
     "question": "Can a warship fly into a star that has exploded?",
     "gold_rules": ["75.0", "75.1"],
     "why": "The rule says 'supernova' and 'ship'; the query says 'star that has exploded' and "
            "'warship'. Zero lexical overlap with the gold passage."},

    {"qid": "Q3", "category": "exception_corner",
     "question": "May a player build new warships at a shipyard while an enemy fleet sits in "
                 "the same system?",
     "gold_rules": ["58.6", "14.0", "14.1"],
     "why": "The restriction is stated twice in different vocabulary — PRODUCING UNITS 58.6 and "
            "BLOCKADED 14.1 — and both carry the 'ground forces can still be produced' exception."},

    {"qid": "Q4", "category": "negation_constraint",
     "question": "Can I take a tactical action on a system I have already claimed this round?",
     "gold_rules": ["5.2"],
     "why": "A prohibition ('cannot activate a system that already contains one of their "
            "command tokens') phrased by the player as a permission question."},

    {"qid": "Q5", "category": "role_limit",
     "question": "How many planetary defence systems can one player have on the same world?",
     "gold_rules": ["54.3", "74.5"],
     "why": "A numeric cap duplicated across PDS and STRUCTURES; 'world' vs 'planet' and "
            "'planetary defence system' vs 'PDS' are both paraphrases."},

    {"qid": "Q6", "category": "exception_corner",
     "question": "Who may remove the token blocking the galaxy's central planet, and what does "
                 "it cost?",
     "gold_rules": ["46.1"],
     "why": "'central planet' means Mecatol Rex and the token is the custodians token; the cost "
            "(six influence) sits inside an 'unless' clause."},

    {"qid": "Q7", "category": "negation_constraint",
     "question": "When is a player forbidden from passing their turn?",
     "gold_rules": ["3.2", "3.4"],
     "why": "'forbidden' vs 'cannot pass until'; the player-count exception for three- and "
            "four-player games is a bullet under the general rule."},

    {"qid": "Q8", "category": "buried_vital_chunk",
     "question": "If a unit takes damage, does it fight less effectively?",
     "gold_rules": ["76.2"],
     "why": "The answer is a negation plus an exception ('functionally the same, except that it "
            "cannot use Sustain Damage') inside a section about marking damage."},

    # Q9 and Q10 were not guessed: they come from the retrieval-only hunt in §4b below, which
    # searched 16 candidate questions for gold chunks that the bi-encoder buries at ranks 4-10.
    {"qid": "Q9", "category": "role_limit",
     "question": "How many swaps am I allowed with the player beside me during one turn?",
     "gold_rules": ["83.1"],
     "why": "The cap lives in TRANSACTIONS 83.1 ('up to one transaction with each of their "
            "neighbors'), but 'swaps' and 'player beside me' pull the bi-encoder towards DEALS "
            "and NEIGHBORS, which define the relationship without stating the limit."},

    {"qid": "Q10", "category": "negation_constraint",
     "question": "What stops my factory from turning out new warships?",
     "gold_rules": ["14.0", "14.1"],
     "why": "'Factory' is not a term the rulebook uses, and the answer is a prohibition in "
            "BLOCKADED 14.1. Contrast with Q3, which asks the same thing in a compound way and "
            "gets no promotion — query specificity, not just vocabulary, moves the ranking."},
]
print(f"{len(QUERIES)} queries; LLM budget {LLM_CALL_BUDGET}, used so far {LLM_CALLS}")

def gold_chunk_ids(gold_rules):
    """Chunk is gold if it contains any gold rule number. Overlap means a rule can appear in two
    chunks, so this is a set, and 'best gold rank' means the best-ranked member of that set."""
    want = set(gold_rules)
    return {int(r.chunk_id) for r in chunks.itertuples()
            if want & set(str(r.rule_nos).split(";"))}

for q in QUERIES:
    q["gold_chunk_ids"] = gold_chunk_ids(q["gold_rules"])
    assert q["gold_chunk_ids"], f'{q["qid"]}: no chunk contains rules {q["gold_rules"]}'
    print(f'{q["qid"]} gold rules {q["gold_rules"]} -> chunks {sorted(q["gold_chunk_ids"])}')

10 queries; LLM budget 22, used so far 1
Q1 gold rules ['2.4'] -> chunks [6]
Q2 gold rules ['75.0', '75.1'] -> chunks [140]
Q3 gold rules ['58.6', '14.0', '14.1'] -> chunks [27, 105]
Q4 gold rules ['5.2'] -> chunks [11]
Q5 gold rules ['54.3', '74.5'] -> chunks [95, 139]
Q6 gold rules ['46.1'] -> chunks [79]
Q7 gold rules ['3.2', '3.4'] -> chunks [8, 9]
Q8 gold rules ['76.2'] -> chunks [141]
Q9 gold rules ['83.1'] -> chunks [160]
Q10 gold rules ['14.0', '14.1'] -> chunks [27]


## 4b. How the failure queries were found (retrieval only, no LLM calls)

The first pass used eight hand-written questions and produced only **one** clean promotion:
five of the eight had the gold chunk at rank 1 or 2 already, so re-ranking had nothing to
repair. Rather than quietly reword those queries until they looked like wins, the search was
made explicit and cheap: 16 candidate questions, each with its gold chunk identified by the
**authoritative sentence** (a regex over the chunk text) instead of a remembered rule number,
scored through both pipelines. Retrieval and re-ranking cost no API quota, so this sweep is
free and fully reproducible.

Q9 and Q10 came out of this table. Everything else stays in it, including the cases where
re-ranking **hurt** or could not help at all — those are findings, not failures to hide.

In [11]:
HUNT_CANDIDATES = [
    ("H1",  "transport",           "If a ship carrying troops is destroyed, what happens to the troops?",
     r"fighters and ground forces being transported.{0,80}removed from the board"),
    ("H2",  "keyword_trap",        "Can I strike a bargain with a player sitting on the far side of the galaxy?",
     r"deals with each other at any time, even if they are not neighbors"),
    ("H3",  "role_limit",          "How many swaps am I allowed with the player beside me during one turn?",
     r"up to one transaction with each of their neighbors"),
    ("H4",  "role_limit",          "Is there a cap on how many victory conditions I can claim in a single round?",
     r"maximum of one public objective and one secret objective"),
    ("H5",  "negation_constraint", "Can my foot soldiers walk from one world to another by themselves?",
     r"Ground forces.{0,120}capacity values, or being transported"),
    ("H6",  "exception_corner",    "What do I have to roll to keep my vessel from being sucked into the rift?",
     r"gravity rift.{0,200}destroyed"),
    ("H7",  "exception_corner",    "At what point is someone knocked out of the game for good?",
     r"A player is eliminated when they meet all"),
    ("H8",  "exception_corner",    "Can I pull my fleet out of a battle once shooting has started?",
     r"ANNOUNCE RETREATS"),
    ("H9",  "keyword_trap",        "What limits the number of warships I can keep in one system?",
     r"fleet pool"),
    ("H10", "negation_constraint", "What stops my factory from turning out new warships?",
     r"blockaded.{0,120}produce ships"),
    ("H11", "exception_corner",    "If two abilities go off at the same instant, which one resolves first?",
     r"abilit(?:y|ies).{0,120}(?:same time|simultaneous)"),
    ("H12", "negation_constraint", "May I hand over a promissory note I already gave away once?",
     r"maximum of one promissory note from their hand"),
    ("H13", "synonym_gap",         "Do wrecked vessels go back into my reserves?",
     r"removed from the game board and returned to their reinforcements"),
    ("H14", "keyword_trap",        "How do I take over a world that already belongs to a rival?",
     r"INVASION|commit ground forces"),
    ("H15", "exception_corner",    "Can the central world be invaded before anyone pays the toll?",
     r"custodians token"),
    ("H16", "negation_constraint", "Am I allowed to activate a system twice in the same turn?",
     r"cannot activate a system that already contains"),
]

hunt = []
for hid, cat, question, gold_pat in HUNT_CANDIDATES:
    gold = {int(r.chunk_id) for r in chunks.itertuples()
            if re.search(gold_pat, str(r.text), re.I)}
    cands = baseline_retrieve(question)
    reranked = rerank(question, cands)
    bi_r = min([c["bi_rank"] for c in cands    if c["chunk_id"] in gold], default=None)
    ce_r = min([c["ce_rank"] for c in reranked if c["chunk_id"] in gold], default=None)

    if not gold:
        verdict = "no gold match (regex too strict)"
    elif bi_r is None:
        verdict = "gold never entered Top 10"          # re-ranking cannot repair this
    elif bi_r > TOP_K_CONTEXT and ce_r <= TOP_K_CONTEXT:
        verdict = "promotion"
    elif bi_r <= TOP_K_CONTEXT and ce_r > TOP_K_CONTEXT:
        verdict = "regression"
    else:
        verdict = "no change in Top 3 membership"
    hunt.append({"hid": hid, "category": cat, "question": question,
                 "n_gold_chunks": len(gold), "gold_chunks": ";".join(map(str, sorted(gold))),
                 "gold_rank_baseline": bi_r, "gold_rank_rerank": ce_r, "verdict": verdict,
                 "promoted_to_queryset": hid in ("H3", "H10")})

hunt_df = pd.DataFrame(hunt)
hunt_df.to_csv(RESULTS / "failure_hunt.csv", index=False)
print(hunt_df[["hid", "category", "gold_rank_baseline", "gold_rank_rerank", "verdict",
               "promoted_to_queryset"]].to_string(index=False))
print("\n" + hunt_df.verdict.value_counts().to_string())
print(f"\n16 candidates screened at zero API cost; "
      f"{(hunt_df.verdict == 'promotion').sum()} promotions, "
      f"{(hunt_df.verdict == 'regression').sum()} regressions.")

hid            category  gold_rank_baseline  gold_rank_rerank                          verdict  promoted_to_queryset
 H1           transport                 NaN               NaN no gold match (regex too strict)                 False
 H2        keyword_trap                 3.0               3.0    no change in Top 3 membership                 False
 H3          role_limit                 8.0               1.0                        promotion                  True
 H4          role_limit                 5.0              10.0    no change in Top 3 membership                 False
 H5 negation_constraint                 4.0               3.0                        promotion                 False
 H6    exception_corner                 NaN               NaN no gold match (regex too strict)                 False
 H7    exception_corner                 6.0               2.0                        promotion                 False
 H8    exception_corner                 1.0               1.0   

## 5. Running both pipelines

One baseline retrieval per query, handed unchanged to the cross-encoder. Both Top 3 sets go to
the same prompt template. Everything is recorded per candidate, so the rank tables below are
derived from data rather than retyped.

In [12]:
runs, summary = [], []

for q in QUERIES:
    candidates = baseline_retrieve(q["question"])               # Pipeline 1
    reranked   = rerank(q["question"], candidates)              # Pipeline 2, same candidates
    assert {c["chunk_id"] for c in candidates} == {c["chunk_id"] for c in reranked}, \
        "pipeline 2 must re-rank the baseline candidates, not re-retrieve"

    base_top3 = [c["chunk_id"] for c in candidates[:TOP_K_CONTEXT]]
    rr_top3   = [c["chunk_id"] for c in reranked[:TOP_K_CONTEXT]]
    gold      = q["gold_chunk_ids"]

    bi_gold = [c["bi_rank"] for c in candidates if c["chunk_id"] in gold]
    ce_gold = [c["ce_rank"] for c in reranked   if c["chunk_id"] in gold]
    best_bi = min(bi_gold) if bi_gold else None                 # None = gold not in Top 10 at all
    best_ce = min(ce_gold) if ce_gold else None

    in3_bi = best_bi is not None and best_bi <= TOP_K_CONTEXT
    in3_ce = best_ce is not None and best_ce <= TOP_K_CONTEXT
    if   in3_ce and not in3_bi: outcome = "promotion"
    elif in3_bi and not in3_ce: outcome = "regression"
    elif base_top3 == rr_top3:  outcome = "no_op_identical"
    elif set(base_top3) == set(rr_top3): outcome = "no_op_reordered"
    else: outcome = "changed_context_same_verdict"

    ans1, p1, hit1, t1, m1 = generate(q["question"], base_top3)     # Answer 1
    ans2, p2, hit2, t2, m2 = generate(q["question"], rr_top3)       # Answer 2
    # A model can run out of quota BETWEEN the two calls of one query — it happened here, with
    # Answer 1 landing on gemini-3.5-flash and Answer 2 failing over to the lite model. Comparing
    # those two would measure the model, not the retrieved context, so Answer 1 is regenerated on
    # the surviving model. Models may still differ ACROSS queries; that is recorded per row.
    if m1 != m2:
        print(f'    {q["qid"]}: model failed over mid-query ({m1} -> {m2}); '
              f"regenerating Answer 1 on {m2}")
        ans1, p1, hit1, t1, m1 = generate(q["question"], base_top3, strict_model=True)
    assert m1 == m2, f'{q["qid"]}: Answer 1 from {m1}, Answer 2 from {m2}'
    # The two prompts may differ ONLY in the interpolated excerpts: masking each prompt's own
    # excerpt texts must reduce both to the same skeleton. This is what "identical prompt
    # structure" means operationally, so it is checked rather than asserted in prose.
    def _skeleton(prompt, cids):
        for cid in cids:
            prompt = prompt.replace(corpus_texts[cid], "<EXCERPT>")
        return prompt
    assert _skeleton(p1, base_top3) == _skeleton(p2, rr_top3), "prompt template diverged"

    for c in reranked:
        runs.append({"qid": q["qid"], "category": q["category"], "chunk_id": c["chunk_id"],
                     "rule_nos": chunks.loc[c["chunk_id"], "rule_nos"],
                     "section_title": chunks.loc[c["chunk_id"], "section_title"],
                     "page_from": chunks.loc[c["chunk_id"], "page_from"],
                     "bi_rank": c["bi_rank"], "bi_score": round(c["bi_score"], 4),
                     "ce_rank": c["ce_rank"], "ce_score": round(c["ce_score"], 4),
                     "is_gold": c["chunk_id"] in gold,
                     "in_baseline_top3": c["chunk_id"] in base_top3,
                     "in_rerank_top3": c["chunk_id"] in rr_top3})

    summary.append({"qid": q["qid"], "category": q["category"], "question": q["question"],
                    "gold_rules": ";".join(q["gold_rules"]),
                    "gold_rank_baseline": best_bi, "gold_rank_rerank": best_ce,
                    "gold_in_top3_baseline": in3_bi, "gold_in_top3_rerank": in3_ce,
                    "outcome": outcome,
                    "baseline_top3": ";".join(map(str, base_top3)),
                    "rerank_top3": ";".join(map(str, rr_top3)),
                    "answer1": ans1, "answer2": ans2, "answer_model": m1,
                    "answers_identical": ans1 == ans2,
                    "a1_cache_hit": hit1, "a2_cache_hit": hit2,
                    "a1_seconds": t1, "a2_seconds": t2})
    print(f'{q["qid"]} {q["category"]:<24} gold rank {str(best_bi):>4} -> {str(best_ce):>4}  '
          f'{outcome}')

runs_df = pd.DataFrame(runs)
summary_df = pd.DataFrame(summary)
runs_df.to_csv(RESULTS / "retrieval_runs.csv", index=False)
summary_df.drop(columns=["answer1", "answer2"]).to_csv(RESULTS / "query_summary.csv", index=False)
print(f"\nLLM calls used: {LLM_CALLS}/{LLM_CALL_BUDGET}  "
      f"(cache hits: {int(summary_df.a1_cache_hit.sum() + summary_df.a2_cache_hit.sum())})")
print(summary_df.outcome.value_counts().to_string())
print("\nanswers per model:")
print(summary_df.answer_model.value_counts().to_string())

Q1 keyword_trap             gold rank    1 ->    1  changed_context_same_verdict


Q2 synonym_gap              gold rank    1 ->    1  changed_context_same_verdict


Q3 exception_corner         gold rank    4 ->    6  changed_context_same_verdict
Q4 negation_constraint      gold rank    6 ->    1  promotion


Q5 role_limit               gold rank    1 ->    1  changed_context_same_verdict
Q6 exception_corner         gold rank None -> None  changed_context_same_verdict


Q7 negation_constraint      gold rank    1 ->    1  no_op_identical
Q8 buried_vital_chunk       gold rank    2 ->    2  changed_context_same_verdict


    Q9: model failed over mid-query (gemini-3.5-flash -> gemini-3.1-flash-lite); regenerating Answer 1 on gemini-3.1-flash-lite


Q9 role_limit               gold rank    8 ->    1  promotion


    transient failure on gemini-3.1-flash-lite (ServerError); waiting 15s (attempt 1/4)


Q10 negation_constraint      gold rank    7 ->    1  promotion

LLM calls used: 4/22  (cache hits: 17)
outcome
changed_context_same_verdict    6
promotion                       3
no_op_identical                 1

answers per model:
answer_model
gemini-3.5-flash         8
gemini-3.1-flash-lite    2


## 6. Rank-shift analysis

For each query, where the best gold chunk sat in the baseline ranking versus after
re-ranking. `None` in the baseline column would mean the gold chunk never entered the Top 10,
which re-ranking cannot fix — the cross-encoder only ever sees those ten candidates.

In [13]:
shift = summary_df[["qid", "category", "gold_rules", "gold_rank_baseline", "gold_rank_rerank",
                    "outcome"]].copy()
# pd.notna, not truthiness: a missing gold rank is NaN, and NaN is truthy.
shift["shift"] = shift.apply(
    lambda r: (f'#{int(r.gold_rank_baseline)} -> #{int(r.gold_rank_rerank)}'
               if pd.notna(r.gold_rank_baseline) and pd.notna(r.gold_rank_rerank)
               else "gold never entered Top 10"), axis=1)
shift.to_csv(RESULTS / "rank_shifts.csv", index=False)
print(shift.to_string(index=False))

# Per-query detail for the promotions: what the cross-encoder pushed down to make room.
for qid in shift.loc[shift.outcome == "promotion", "qid"]:
    d = runs_df[runs_df.qid == qid].sort_values("ce_rank")
    print(f"\n=== {qid}: {summary_df.set_index('qid').loc[qid, 'question']}")
    print(d[["bi_rank", "ce_rank", "rule_nos", "section_title", "page_from", "bi_score",
             "ce_score", "is_gold"]].to_string(index=False))

qid            category     gold_rules  gold_rank_baseline  gold_rank_rerank                      outcome                     shift
 Q1        keyword_trap            2.4                 1.0               1.0 changed_context_same_verdict                  #1 -> #1
 Q2         synonym_gap      75.0;75.1                 1.0               1.0 changed_context_same_verdict                  #1 -> #1
 Q3    exception_corner 58.6;14.0;14.1                 4.0               6.0 changed_context_same_verdict                  #4 -> #6
 Q4 negation_constraint            5.2                 6.0               1.0                    promotion                  #6 -> #1
 Q5          role_limit      54.3;74.5                 1.0               1.0 changed_context_same_verdict                  #1 -> #1
 Q6    exception_corner           46.1                 NaN               NaN changed_context_same_verdict gold never entered Top 10
 Q7 negation_constraint        3.2;3.4                 1.0               1.0

## 7. Near-duplicate candidates from chunk overlap

Chunks overlap by one rule unit, so a rule can appear in two adjacent chunks. When both land
in the Top 10 they consume two of the ten candidate slots (and, if both reach the Top 3, two
of the three context slots) with nearly the same text. Logged rather than silently tolerated,
because it is a direct cost of the overlap decision.

In [14]:
dups = []
for qid, grp in runs_df.groupby("qid"):
    ids = grp.chunk_id.tolist()
    in_top3 = dict(zip(grp.chunk_id, grp.in_baseline_top3))
    for i, a in enumerate(ids):
        for b in ids[i + 1:]:
            cos = float(corpus_emb[a] @ corpus_emb[b])
            if cos >= NEAR_DUP_COSINE:
                shared = set(str(chunks.loc[a, "rule_nos"]).split(";")) & \
                         set(str(chunks.loc[b, "rule_nos"]).split(";"))
                dups.append({"qid": qid, "chunk_a": a, "chunk_b": b, "cosine": round(cos, 4),
                             "shared_rules": ";".join(sorted(shared)) or "(none)",
                             "cause": "chunk overlap" if shared else "document repeats itself",
                             "both_in_baseline_top3": bool(in_top3[a] and in_top3[b])})
dups_df = pd.DataFrame(dups)
dups_df.to_csv(RESULTS / "near_duplicates.csv", index=False)
print(f"near-duplicate pairs in Top 10 (cosine >= {NEAR_DUP_COSINE}): {len(dups_df)}")
if len(dups_df):
    print(dups_df.to_string(index=False))

near-duplicate pairs in Top 10 (cosine >= 0.9): 7
qid  chunk_a  chunk_b  cosine shared_rules         cause  both_in_baseline_top3
 Q1      137      138  0.9318         73.1 chunk overlap                  False
Q10       83       84  0.9301         49.4 chunk overlap                  False
 Q3       44       45  0.9342         23.3 chunk overlap                  False
 Q5       55       54  0.9414         30.2 chunk overlap                  False
 Q7        9        8  0.9058          3.3 chunk overlap                   True
 Q9        9        8  0.9058          3.3 chunk overlap                   True
 Q9      137      138  0.9318         73.1 chunk overlap                  False


## 8. Answer 1 vs Answer 2

In [15]:
models_used = ", ".join(f"`{m}` ({n})" for m, n in summary_df.answer_model.value_counts().items())
lines = ["# Answer 1 (baseline) vs Answer 2 (re-ranked)", "",
         f"Models: {models_used} · temperature: provider default (see NOTES.md 1.2) · "
         f"thinking: minimal · identical prompt template for both answers. Both answers of any "
         f"one query always come from the same model (asserted in §5).", ""]
for r in summary_df.itertuples():
    lines += [f"## {r.qid} — {r.category}", "", f"**Q:** {r.question}", "",
              f"*Model:* `{r.answer_model}` · *gold rules:* `{r.gold_rules}` · *gold rank:* "
              f"`#{r.gold_rank_baseline}` → `#{r.gold_rank_rerank}` · *outcome:* `{r.outcome}` · "
              f"*answers identical:* `{r.answers_identical}`", "",
              f"*Baseline Top 3 (chunk ids):* `{r.baseline_top3}` · "
              f"*Re-ranked Top 3:* `{r.rerank_top3}`", "",
              "**Answer 1**", "", r.answer1, "", "**Answer 2**", "", r.answer2, "", "---", ""]
(RESULTS / "answers.md").write_text("\n".join(lines))
print(f'wrote {RESULTS / "answers.md"} ({len(lines)} lines)')

for r in summary_df.itertuples():
    if not r.answers_identical:
        print(f"\n=== {r.qid} ({r.outcome}) {r.question}")
        print(f"--- Answer 1 (baseline top3 {r.baseline_top3}):\n{r.answer1}")
        print(f"--- Answer 2 (reranked top3 {r.rerank_top3}):\n{r.answer2}")

wrote /Users/d400l/Desktop/LMA_RAG/LMA_RAG/results/answers.md (184 lines)

=== Q1 (changed_context_same_verdict) How many cards is a player allowed to keep in their hand between rounds?
--- Answer 1 (baseline top3 6;137;91):
According to Rule 2.4, a player's hand can have a maximum of seven action cards, though a game effect can increase or decrease this limit. If a player ever has more than seven action cards, they must choose seven cards to keep and discard the rest (Rule 2.4).

**Rules cited:** Rule 2.4
--- Answer 2 (reranked top3 6;57;70):
According to Rule 2.4, a player’s hand can have a maximum of seven action cards, and if a player ever has more than seven, they must choose seven to keep and discard the rest. However, game effects can increase or decrease this hand limit. 

[Rule 2.4]

=== Q2 (changed_context_same_verdict) Can a warship fly into a star that has exploded?
--- Answer 1 (baseline top3 140;115;145):
No, a ship cannot move into a supernova (a star that has exploded).

## 9. Latency profiling

Each stage is timed on its own, `LATENCY_REPEATS` times per query, after a discarded warm-up
pass — the first call loads weights and allocates buffers, and including it would swamp the
numbers. Hardware and device are printed with the table because these are wall-clock figures
from one laptop, not published benchmarks.

In [16]:
def time_ms(fn, n=LATENCY_REPEATS):
    fn()                                                  # warm-up, discarded
    out = []
    for _ in range(n):
        t0 = time.perf_counter(); fn(); out.append((time.perf_counter() - t0) * 1000)
    return out

timings = {"1. query embedding (bi-encoder)": [], "2. FAISS search (top 10 of N)": [],
           "3. cross-encoder scoring (10 pairs)": [],
           "Pipeline 1 retrieval total (1+2)": [],
           "Pipeline 2 retrieval total (1+2+3)": []}

for q in QUERIES:
    question = q["question"]
    qvec = bi_encoder.encode([question], convert_to_numpy=True,
                             normalize_embeddings=True).astype(np.float32)
    cands = baseline_retrieve(question)
    pairs = [(question, corpus_texts[c["chunk_id"]]) for c in cands]

    timings["1. query embedding (bi-encoder)"] += time_ms(
        lambda: bi_encoder.encode([question], convert_to_numpy=True, normalize_embeddings=True))
    timings["2. FAISS search (top 10 of N)"] += time_ms(
        lambda: index.search(qvec, TOP_K_RETRIEVE))
    timings["3. cross-encoder scoring (10 pairs)"] += time_ms(
        lambda: cross_encoder.predict(pairs, show_progress_bar=False))
    timings["Pipeline 1 retrieval total (1+2)"] += time_ms(
        lambda: baseline_retrieve(question))
    timings["Pipeline 2 retrieval total (1+2+3)"] += time_ms(
        lambda: rerank(question, baseline_retrieve(question)))

lat_rows = []
for stage, vals in timings.items():
    lat_rows.append({"stage": stage, "n": len(vals),
                     "mean_ms": round(statistics.mean(vals), 2),
                     "std_ms": round(statistics.stdev(vals), 2) if len(vals) > 1 else 0.0,
                     "median_ms": round(statistics.median(vals), 2),
                     "min_ms": round(min(vals), 2), "max_ms": round(max(vals), 2)})

# LLM latency, from the uncached generations only. Cache hits are excluded rather than averaged
# in, which would quietly divide the real cost by the hit rate.
llm_times = [t * 1000 for t in list(summary_df.a1_seconds) + list(summary_df.a2_seconds) if t > 0]
if llm_times:
    lat_rows.append({"stage": "LLM generation (uncached calls only)", "n": len(llm_times),
                     "mean_ms": round(statistics.mean(llm_times), 2),
                     "std_ms": round(statistics.stdev(llm_times), 2) if len(llm_times) > 1 else 0.0,
                     "median_ms": round(statistics.median(llm_times), 2),
                     "min_ms": round(min(llm_times), 2), "max_ms": round(max(llm_times), 2)})

latency = pd.DataFrame(lat_rows)
latency["hardware"] = HARDWARE
latency.to_csv(RESULTS / "latency.csv", index=False)

print(f"hardware: {HARDWARE}")
print(f"corpus: {len(chunks)} chunks | queries: {len(QUERIES)} | repeats per query: "
      f"{LATENCY_REPEATS}\n")
print(latency.drop(columns=["hardware"]).to_string(index=False))

p1 = latency.loc[latency.stage.str.startswith("Pipeline 1"), "mean_ms"].iloc[0]
p2 = latency.loc[latency.stage.str.startswith("Pipeline 2"), "mean_ms"].iloc[0]
print(f"\nre-ranking adds {p2 - p1:.1f} ms ({p2 / p1:.2f}x) to retrieval for "
      f"{TOP_K_RETRIEVE} candidates")

hardware: arm64 / Darwin 25.4.0 / torch 2.5.1 / device=mps
corpus: 172 chunks | queries: 10 | repeats per query: 10

                               stage   n  mean_ms  std_ms  median_ms  min_ms   max_ms
     1. query embedding (bi-encoder) 100    15.71    0.96      15.50   14.61    20.58
       2. FAISS search (top 10 of N) 100     0.02    0.01       0.01    0.01     0.06
 3. cross-encoder scoring (10 pairs) 100   151.54   20.66     147.04  107.73   244.19
    Pipeline 1 retrieval total (1+2) 100    16.11    1.30      15.71   14.81    21.71
  Pipeline 2 retrieval total (1+2+3) 100   169.72   23.62     161.78  125.95   252.65
LLM generation (uncached calls only)   3  8603.36 4550.87    8625.25 4041.58 13143.24

re-ranking adds 153.6 ms (10.54x) to retrieval for 10 candidates


## 10. Why not cross-encode the whole collection?

Measured rather than asserted: the cross-encoder's throughput on this machine, extrapolated to
the full corpus and to a million-chunk collection, against what the bi-encoder path actually
costs per query.

In [17]:
# Score the ENTIRE corpus against one query — literally the thing this section argues against.
BULK = len(chunks)
bulk_pairs = [(QUERIES[0]["question"], t) for t in corpus_texts]
bulk_ms = time_ms(lambda: cross_encoder.predict(bulk_pairs, show_progress_bar=False), n=3)
pairs_per_sec = BULK / (statistics.mean(bulk_ms) / 1000)

embed_ms  = latency.loc[latency.stage.str.startswith("1."), "mean_ms"].iloc[0]
search_ms = latency.loc[latency.stage.str.startswith("2."), "mean_ms"].iloc[0]

scale_rows = []
for n_chunks in (len(chunks), 10_000, 1_000_000):
    ce_all = n_chunks / pairs_per_sec
    scale_rows.append({
        "corpus_chunks": n_chunks,
        "cross_encode_entire_corpus_s": round(ce_all, 2),
        "baseline_retrieval_s": round((embed_ms + search_ms) / 1000, 5),
        "rerank_top10_s": round(p2 / 1000, 5),
        "ce_all_vs_rerank_top10_x": round(ce_all / (p2 / 1000), 1)})
scaling = pd.DataFrame(scale_rows)
scaling["pairs_per_sec"] = round(pairs_per_sec, 1)
scaling["hardware"] = HARDWARE
scaling.to_csv(RESULTS / "scaling.csv", index=False)

print(f"cross-encoder throughput: {pairs_per_sec:.1f} (query, chunk) pairs/sec "
      f"[{BULK} pairs, mean of 3 runs, device={DEVICE}]\n")
print(scaling.drop(columns=["hardware"]).to_string(index=False))
print(f"\nThe bi-encoder embeds each chunk ONCE at index time; the cross-encoder cannot reuse "
      f"anything,\nbecause its score depends on the query and the chunk jointly. Its cost is "
      f"therefore O(N) per query,\nwhile FAISS search is sublinear and the two-stage pipeline "
      f"keeps the O(N) stage at N={TOP_K_RETRIEVE}.")

cross-encoder throughput: 71.0 (query, chunk) pairs/sec [172 pairs, mean of 3 runs, device=mps]

 corpus_chunks  cross_encode_entire_corpus_s  baseline_retrieval_s  rerank_top10_s  ce_all_vs_rerank_top10_x  pairs_per_sec
           172                          2.42               0.01573         0.16972                      14.3           71.0
         10000                        140.90               0.01573         0.16972                     830.2           71.0
       1000000                      14089.75               0.01573         0.16972                   83017.6           71.0

The bi-encoder embeds each chunk ONCE at index time; the cross-encoder cannot reuse anything,
because its score depends on the query and the chunk jointly. Its cost is therefore O(N) per query,
while FAISS search is sublinear and the two-stage pipeline keeps the O(N) stage at N=10.


## 11. Outputs

| file | contents |
|---|---|
| `results/chunks.csv` | the corpus: chunk id, section, rule numbers, pages, token count, text |
| `results/retrieval_runs.csv` | every candidate of every query with both ranks and both scores |
| `results/query_summary.csv` | one row per query: gold ranks, outcome, Top 3 sets |
| `results/rank_shifts.csv` | gold-chunk rank before and after re-ranking |
| `results/near_duplicates.csv` | overlap-induced and document-level duplicate candidates |
| `results/answers.md` | Answer 1 vs Answer 2, side by side |
| `results/answers_cache.json` | frozen generations keyed by (model, prompt hash) |
| `results/latency.csv` | per-stage timings, mean ± std, with hardware |
| `results/scaling.csv` | cross-encoder throughput and the extrapolation |

`NOTES.md` records why each decision was made and what was observed, including the cases where
re-ranking changed nothing. The report is written by hand from these files.